In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.regression.linear_model import OLS
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# PART 1: SAMPLE DATA STRUCTURE (Replace with your actual data)
# =============================================================================

np.random.seed(42)

# Parameters
n_ppgs = 10
n_weeks = 104
ppg_ids = [f'PPG_{str(i).zfill(2)}' for i in range(1, n_ppgs + 1)]

# Create base dataframe
rows = []
for ppg in ppg_ids:
    for week in range(1, n_weeks + 1):
        rows.append({'ppg_id': ppg, 'week_num': week})

df = pd.DataFrame(rows)

# Add week_end date
df['week_end'] = pd.to_datetime('2022-01-01') + pd.to_timedelta((df['week_num'] - 1) * 7, unit='D')

# Simulate realistic data (REPLACE THIS SECTION WITH YOUR ACTUAL DATA)
# -----------------------------------------------------------------------------

# PPG-specific base characteristics
ppg_chars = {
    'PPG_01': {'base_price': 9.49, 'base_units': 45000, 'comp_name': 'Folgers_Premium', 'comp_base_price': 8.99},
    'PPG_02': {'base_price': 12.99, 'base_units': 32000, 'comp_name': 'Folgers_Classic', 'comp_base_price': 11.49},
    'PPG_03': {'base_price': 7.49, 'base_units': 58000, 'comp_name': 'Maxwell_House', 'comp_base_price': 6.99},
    'PPG_04': {'base_price': 14.99, 'base_units': 22000, 'comp_name': 'Peets_Major', 'comp_base_price': 14.49},
    'PPG_05': {'base_price': 8.99, 'base_units': 51000, 'comp_name': 'Starbucks_House', 'comp_base_price': 9.99},
    'PPG_06': {'base_price': 11.49, 'base_units': 28000, 'comp_name': 'Dunkin_Original', 'comp_base_price': 10.99},
    'PPG_07': {'base_price': 6.99, 'base_units': 65000, 'comp_name': 'Store_Brand', 'comp_base_price': 5.99},
    'PPG_08': {'base_price': 15.99, 'base_units': 18000, 'comp_name': 'Lavazza_Super', 'comp_base_price': 16.49},
    'PPG_09': {'base_price': 10.49, 'base_units': 38000, 'comp_name': 'Folgers_1850', 'comp_base_price': 9.99},
    'PPG_10': {'base_price': 9.99, 'base_units': 42000, 'comp_name': 'McCafe_Premium', 'comp_base_price': 8.49},
}

# Add PPG characteristics
df['comp_name'] = df['ppg_id'].map(lambda x: ppg_chars[x]['comp_name'])
base_price = df['ppg_id'].map(lambda x: ppg_chars[x]['base_price'])
base_units = df['ppg_id'].map(lambda x: ppg_chars[x]['base_units'])
comp_base = df['ppg_id'].map(lambda x: ppg_chars[x]['comp_base_price'])

# Simulate price variation (yours will be actual data)
df['own_base_price'] = base_price * (1 + np.random.uniform(-0.08, 0.12, len(df)))
df['comp_price'] = comp_base * (1 + np.random.uniform(-0.05, 0.10, len(df)))

# Simulate TDP (distribution)
df['tdp'] = np.random.uniform(60, 95, len(df))

# Simulate base_unit_pct (% of units that are base vs incremental/promo)
df['base_unit_pct'] = np.random.uniform(0.75, 0.95, len(df))

# Simulate base units with realistic relationships
# (In your case, this is your actual dependent variable)
seasonality = 1 + 0.15 * np.sin(2 * np.pi * df['week_num'] / 52)
price_effect = (df['own_base_price'] / base_price) ** -2.5  # elasticity ~ -2.5
comp_effect = (df['comp_price'] / comp_base) ** 0.8  # cross-elasticity ~ +0.8
tdp_effect = (df['tdp'] / 80) ** 0.7
noise = np.random.lognormal(0, 0.08, len(df))

df['base_units'] = base_units * seasonality * price_effect * comp_effect * tdp_effect * noise

print("="*70)
print("PART 1: RAW DATA STRUCTURE")
print("="*70)
print(f"\nDataset shape: {df.shape}")
print(f"PPGs: {df['ppg_id'].nunique()}")
print(f"Weeks per PPG: {df.groupby('ppg_id').size().iloc[0]}")
print("\nSample rows:")
print(df[['week_end', 'ppg_id', 'own_base_price', 'comp_name', 'comp_price', 'tdp', 'base_unit_pct', 'base_units']].head(10))

PART 1: RAW DATA STRUCTURE

Dataset shape: (1040, 9)
PPGs: 10
Weeks per PPG: 104

Sample rows:
    week_end  ppg_id  own_base_price        comp_name  comp_price        tdp  \
0 2022-01-01  PPG_01        9.441677  Folgers_Premium    9.268004  60.950860   
1 2022-01-08  PPG_01       10.535256  Folgers_Premium    9.606790  62.282161   
2 2022-01-15  PPG_01       10.120125  Folgers_Premium    8.970338  76.237598   
3 2022-01-22  PPG_01        9.867054  Folgers_Premium    9.384515  91.822707   
4 2022-01-29  PPG_01        9.026923  Folgers_Premium    9.735241  78.854563   
5 2022-02-05  PPG_01        9.026878  Folgers_Premium    9.370992  77.423438   
6 2022-02-12  PPG_01        8.841043  Folgers_Premium    8.854646  63.691579   
7 2022-02-19  PPG_01       10.374802  Folgers_Premium    8.573404  82.987304   
8 2022-02-26  PPG_01        9.871716  Folgers_Premium    9.713828  88.773611   
9 2022-03-05  PPG_01       10.074722  Folgers_Premium    8.569182  73.314700   

   base_unit_pct    base

In [3]:
df.head()

,ppg_id,week_num,week_end,comp_name,own_base_price,comp_price,tdp,base_unit_pct,base_units
0,PPG_01,1,2022-01-01,Folgers_Premium,9.441677,9.268004,60.950860,0.804161,36686.204319
1,PPG_01,2,2022-01-08,Folgers_Premium,10.535256,9.606790,62.282161,0.799512,31847.249451
2,PPG_01,3,2022-01-15,Folgers_Premium,10.120125,8.970338,76.237598,0.762505,32789.417971
3,PPG_01,4,2022-01-22,Folgers_Premium,9.867054,9.384515,91.822707,0.841788,53388.743302
4,PPG_01,5,2022-01-29,Folgers_Premium,9.026923,9.735241,78.854563,0.896547,51531.869272


In [4]:
df.shape

(1040, 9)

In [6]:
print("\n" + "="*70)
print("PART 2: FEATURE ENGINEERING")
print("="*70)

# 2.1 Log transformations (for multiplicative/elasticity model)
df['ln_base_units'] = np.log(df['base_units'])
df['ln_own_price'] = np.log(df['own_base_price'])
df['ln_comp_price'] = np.log(df['comp_price'])
df['ln_tdp'] = np.log(df['tdp'])

# 2.2 Price gap (alternative to log comp price - use one or the other)
df['price_gap'] = df['own_base_price'] - df['comp_price']
df['price_index'] = df['own_base_price'] / df['comp_price']

# 2.3 Trend
df['trend'] = df.groupby('ppg_id').cumcount() + 1

# 2.4 Seasonality - Fourier terms (more parsimonious than monthly dummies)
df['sin_52'] = np.sin(2 * np.pi * df['week_num'] / 52)
df['cos_52'] = np.cos(2 * np.pi * df['week_num'] / 52)
# Second harmonic for more complex seasonality (optional)
df['sin_26'] = np.sin(2 * np.pi * df['week_num'] / 26)
df['cos_26'] = np.cos(2 * np.pi * df['week_num'] / 26)

# 2.5 Holiday flags (customize based on your data)
# Map week_end to identify holiday weeks
df['month'] = df['week_end'].dt.month
df['day'] = df['week_end'].dt.day

# Simple holiday flags (adjust dates for your specific weeks)
df['holiday_thanksgiving'] = ((df['month'] == 11) & (df['day'] >= 20) & (df['day'] <= 30)).astype(int)
df['holiday_christmas'] = ((df['month'] == 12) & (df['day'] >= 18)).astype(int)
df['holiday_july4'] = ((df['month'] == 7) & (df['day'] >= 1) & (df['day'] <= 7)).astype(int)
df['holiday_memorial'] = ((df['month'] == 5) & (df['day'] >= 25)).astype(int)
df['holiday_labor'] = ((df['month'] == 9) & (df['day'] <= 7)).astype(int)

# Combined holiday flag (or keep separate if effects differ)
df['is_holiday'] = (df['holiday_thanksgiving'] | df['holiday_christmas'] | 
                    df['holiday_july4'] | df['holiday_memorial'] | df['holiday_labor']).astype(int)

# 2.6 Lag features (competitor reaction - 2-3 week lag you mentioned)
df = df.sort_values(['ppg_id', 'week_num'])
df['ln_comp_price_lag2'] = df.groupby('ppg_id')['ln_comp_price'].shift(2)
df['ln_own_price_lag2'] = df.groupby('ppg_id')['ln_own_price'].shift(2)

# Drop rows with NaN from lagging (first 2 weeks per PPG)
df_model = df.dropna().copy()

print("\nFeature columns created:")
feature_cols = ['ln_base_units', 'ln_own_price', 'ln_comp_price', 'ln_tdp', 
                'price_gap', 'price_index', 'trend', 'sin_52', 'cos_52', 
                'sin_26', 'cos_26', 'is_holiday', 'ln_comp_price_lag2']
print(feature_cols)

print("\nModel-ready data sample:")
print(df_model[['ppg_id', 'week_end', 'ln_base_units', 'ln_own_price', 'ln_comp_price', 
                'ln_tdp', 'base_unit_pct', 'trend', 'sin_52', 'is_holiday']].head(10))



PART 2: FEATURE ENGINEERING

Feature columns created:
['ln_base_units', 'ln_own_price', 'ln_comp_price', 'ln_tdp', 'price_gap', 'price_index', 'trend', 'sin_52', 'cos_52', 'sin_26', 'cos_26', 'is_holiday', 'ln_comp_price_lag2']

Model-ready data sample:
    ppg_id   week_end  ln_base_units  ln_own_price  ln_comp_price    ln_tdp  \
2   PPG_01 2022-01-15      10.397861      2.314526       2.193923  4.333855   
3   PPG_01 2022-01-22      10.885355      2.289201       2.239061  4.519860   
4   PPG_01 2022-01-29      10.849956      2.200212       2.275752  4.367605   
5   PPG_01 2022-02-05      11.064887      2.200207       2.237619  4.349290   
6   PPG_01 2022-02-12      10.799783      2.179405       2.180942  4.154052   
7   PPG_01 2022-02-19      10.578737      2.339380       2.148665  4.418688   
8   PPG_01 2022-02-26      10.825114      2.289674       2.273550  4.486089   
9   PPG_01 2022-03-05      10.573709      2.310029       2.148172  4.294761   
10  PPG_01 2022-03-12      11.2835

In [9]:
df_model

,ppg_id,week_num,week_end,comp_name,own_base_price,comp_price,tdp,base_unit_pct,base_units,ln_base_units,...,month,day,holiday_thanksgiving,holiday_christmas,holiday_july4,holiday_memorial,holiday_labor,is_holiday,ln_comp_price_lag2,ln_own_price_lag2
2,PPG_01,3,2022-01-15,Folgers_Premium,10.120125,8.970338,76.237598,0.762505,32789.417971,10.397861,...,1,15,0,0,0,0,0,0,2.226568,2.245134
3,PPG_01,4,2022-01-22,Folgers_Premium,9.867054,9.384515,91.822707,0.841788,53388.743302,10.885355,...,1,22,0,0,0,0,0,0,2.262470,2.354727
4,PPG_01,5,2022-01-29,Folgers_Premium,9.026923,9.735241,78.854563,0.896547,51531.869272,10.849956,...,1,29,0,0,0,0,0,0,2.193923,2.314526
5,PPG_01,6,2022-02-05,Folgers_Premium,9.026878,9.370992,77.423438,0.871346,63888.026614,11.064887,...,2,5,0,0,0,0,0,0,2.239061,2.289201
6,PPG_01,7,2022-02-12,Folgers_Premium,8.841043,8.854646,63.691579,0.884574,49010.170072,10.799783,...,2,12,0,0,0,0,0,0,2.275752,2.200212
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1035,PPG_10,100,2023-11-25,McCafe_Premium,10.581985,9.174182,90.711452,0.752579,42603.161329,10.659684,...,11,25,1,0,0,0,0,1,2.217399,2.244076
1036,PPG_10,101,2023-12-02,McCafe_Premium,9.869870,8.369607,85.855310,0.858060,38903.225057,10.568832,...,12,2,0,0,0,0,0,0,2.217642,2.234794
1037,PPG_10,102,2023-12-09,McCafe_Premium,10.638884,8.640153,60.481094,0.920230,31571.516204,10.360011,...,12,9,0,0,0,0,0,0,2.216393,2.359153
1038,PPG_10,103,2023-12-16,McCafe_Premium,9.321382,9.319884,68.693169,0.941526,48082.224768,10.780668,...,12,16,0,0,0,0,0,0,2.124607,2.289487


In [19]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Check VIF for key variables
features_to_check = ['ln_own_price', 'ln_comp_price', 'ln_tdp', 'base_unit_pct', 'trend']
X = df_model[features_to_check].copy()
X['const'] = 1

vif_data = pd.DataFrame({
    'feature': features_to_check,
    'VIF': [variance_inflation_factor(X.values, i) for i in range(len(features_to_check))]
})
print(vif_data)

         feature       VIF
0   ln_own_price  8.062792
1  ln_comp_price  8.057608
2         ln_tdp  1.002300
3  base_unit_pct  1.000273
4          trend  1.000050


In [8]:
print("\n" + "="*70)
print("PART 3: POOLED MODEL WITH PPG-SPECIFIC COEFFICIENTS")
print("="*70)

# -----------------------------------------------------------------------------
# MODEL SPECIFICATION:
# - PPG-specific intercepts (fixed effects)
# - PPG-specific own-price elasticity (interaction: ln_own_price:ppg_id)
# - PPG-specific cross-price elasticity (interaction: ln_comp_price:ppg_id)
# - PPG-specific TDP elasticity (interaction: ln_tdp:ppg_id)
# - Shared seasonality, trend, holiday effects
# - Shared base_unit_pct effect
# -----------------------------------------------------------------------------

# Convert ppg_id to categorical for proper interaction handling
df_model['ppg_id'] = pd.Categorical(df_model['ppg_id'])

# Formula with interactions
# C(ppg_id) creates dummy variables (fixed effects)
# var:C(ppg_id) creates interaction terms (PPG-specific slopes)

formula = """
ln_base_units ~ 
    C(ppg_id) 
    + ln_own_price:C(ppg_id) 
    + ln_comp_price:C(ppg_id) 
    + ln_tdp:C(ppg_id)
    + base_unit_pct
    + trend
    + sin_52 + cos_52
    + sin_26 + cos_26
    + is_holiday
    - 1
"""
# Note: -1 removes global intercept since we have PPG fixed effects

# Fit the model
model = smf.ols(formula=formula, data=df_model).fit()

print("\nModel Summary:")
print(f"R-squared: {model.rsquared:.4f}")
print(f"Adj R-squared: {model.rsquared_adj:.4f}")
print(f"Observations: {model.nobs:.0f}")
print(f"Parameters: {len(model.params)}")



PART 3: POOLED MODEL WITH PPG-SPECIFIC COEFFICIENTS

Model Summary:
R-squared: 0.9698
Adj R-squared: 0.9684
Observations: 1020
Parameters: 47


In [20]:
### mixed effect models 
import statsmodels.formula.api as smf

# CURRENT APPROACH (Fixed Effects / No Pooling on elasticities)
# Each PPG elasticity estimated independently
formula_fixed = """
ln_base_units ~ 
    C(ppg_id) 
    + ln_own_price:C(ppg_id) 
    + ln_comp_price:C(ppg_id) 
    + ln_tdp:C(ppg_id)
    + base_unit_pct + trend + sin_52 + cos_52 + is_holiday
    - 1
"""
model_fixed = smf.ols(formula_fixed, data=df_model).fit()


# PARTIAL POOLING APPROACH (Mixed Effects / Random Slopes)
# PPG elasticities shrunk toward global mean
formula_mixed = """
ln_base_units ~ 
    ln_own_price + ln_comp_price + ln_tdp
    + base_unit_pct + trend + sin_52 + cos_52 + is_holiday
"""

model_mixed = smf.mixedlm(
    formula=formula_mixed,
    data=df_model,
    groups=df_model['ppg_id'],  # Grouping variable
    re_formula='~ln_own_price + ln_comp_price + ln_tdp'  # Random slopes
).fit()

# Extract PPG-specific elasticities
# Fixed effect (global mean) + Random effect (PPG deviation)
print("Global mean elasticities:")
print(model_mixed.fe_params[['ln_own_price', 'ln_comp_price', 'ln_tdp']])

print("\nPPG-specific deviations from mean:")
print(model_mixed.random_effects)  # Dictionary of PPG deviations

Global mean elasticities:
ln_own_price    -2.490350
ln_comp_price    0.791294
ln_tdp           0.698281
dtype: float64

PPG-specific deviations from mean:
{'PPG_01': Group            0.018585
ln_own_price     0.003656
ln_comp_price   -0.008891
ln_tdp           0.004076
dtype: float64, 'PPG_02': Group            0.105710
ln_own_price     0.085967
ln_comp_price   -0.028989
ln_tdp           0.003679
dtype: float64, 'PPG_03': Group           -0.050063
ln_own_price    -0.020155
ln_comp_price    0.084745
ln_tdp          -0.041862
dtype: float64, 'PPG_04': Group            0.006064
ln_own_price     0.023014
ln_comp_price    0.055391
ln_tdp          -0.032305
dtype: float64, 'PPG_05': Group           -0.061662
ln_own_price    -0.012084
ln_comp_price    0.029616
ln_tdp          -0.013863
dtype: float64, 'PPG_06': Group           -0.042215
ln_own_price    -0.050692
ln_comp_price   -0.056376
ln_tdp           0.036598
dtype: float64, 'PPG_07': Group           -0.009187
ln_own_price    -0.050286
ln

In [12]:
model.params

C(ppg_id)[PPG_01]                  12.054863
C(ppg_id)[PPG_02]                  12.170490
C(ppg_id)[PPG_03]                  11.175152
C(ppg_id)[PPG_04]                  11.260964
C(ppg_id)[PPG_05]                  10.063330
C(ppg_id)[PPG_06]                  11.556956
C(ppg_id)[PPG_07]                  11.740682
C(ppg_id)[PPG_08]                  11.771389
C(ppg_id)[PPG_09]                  11.223657
C(ppg_id)[PPG_10]                  11.623212
ln_own_price:C(ppg_id)[PPG_01]     -2.568681
ln_own_price:C(ppg_id)[PPG_02]     -2.448001
ln_own_price:C(ppg_id)[PPG_03]     -2.541433
ln_own_price:C(ppg_id)[PPG_04]     -2.451740
ln_own_price:C(ppg_id)[PPG_05]     -2.265507
ln_own_price:C(ppg_id)[PPG_06]     -2.535964
ln_own_price:C(ppg_id)[PPG_07]     -2.701672
ln_own_price:C(ppg_id)[PPG_08]     -2.433459
ln_own_price:C(ppg_id)[PPG_09]     -2.493236
ln_own_price:C(ppg_id)[PPG_10]     -2.476140
ln_comp_price:C(ppg_id)[PPG_01]     0.650316
ln_comp_price:C(ppg_id)[PPG_02]     0.545908
ln_comp_pr

In [11]:
print("\n" + "="*70)
print("PART 4: PPG-SPECIFIC ELASTICITIES")
print("="*70)

# Extract coefficients into a cleaner format
coefs = model.params
pvalues = model.pvalues

# Parse coefficient names to extract PPG-specific values
elasticity_data = []

for ppg in ppg_ids:
    row = {'ppg_id': ppg}
    
    # Find own-price elasticity for this PPG
    own_price_key = [k for k in coefs.index if 'ln_own_price' in k and ppg in k]
    if own_price_key:
        row['own_price_elasticity'] = coefs[own_price_key[0]]
        row['own_price_pvalue'] = pvalues[own_price_key[0]]
    
    # Find cross-price elasticity for this PPG
    comp_price_key = [k for k in coefs.index if 'ln_comp_price' in k and ppg in k]
    if comp_price_key:
        row['cross_price_elasticity'] = coefs[comp_price_key[0]]
        row['cross_price_pvalue'] = pvalues[comp_price_key[0]]
    
    # Find TDP elasticity for this PPG
    tdp_key = [k for k in coefs.index if 'ln_tdp' in k and ppg in k]
    if tdp_key:
        row['tdp_elasticity'] = coefs[tdp_key[0]]
        row['tdp_pvalue'] = pvalues[tdp_key[0]]
    
    # Find intercept for this PPG
    intercept_key = [k for k in coefs.index if ppg in k and 'ln_' not in k]
    if intercept_key:
        row['intercept'] = coefs[intercept_key[0]]
    
    elasticity_data.append(row)

elasticity_df = pd.DataFrame(elasticity_data)

# Add significance flags
elasticity_df['own_price_sig'] = elasticity_df['own_price_pvalue'].apply(
    lambda x: '***' if x < 0.01 else ('**' if x < 0.05 else ('*' if x < 0.10 else ''))
)
elasticity_df['cross_price_sig'] = elasticity_df['cross_price_pvalue'].apply(
    lambda x: '***' if x < 0.01 else ('**' if x < 0.05 else ('*' if x < 0.10 else ''))
)
elasticity_df['tdp_sig'] = elasticity_df['tdp_pvalue'].apply(
    lambda x: '***' if x < 0.01 else ('**' if x < 0.05 else ('*' if x < 0.10 else ''))
)

print("\nPPG-Specific Elasticities:")
print("-" * 90)
display_cols = ['ppg_id', 'own_price_elasticity', 'own_price_sig', 
                'cross_price_elasticity', 'cross_price_sig', 
                'tdp_elasticity', 'tdp_sig']
print(elasticity_df[display_cols].to_string(index=False))

# Shared coefficients
print("\n\nShared Coefficients (Same for all PPGs):")
print("-" * 50)
shared_vars = ['base_unit_pct', 'trend', 'sin_52', 'cos_52', 'sin_26', 'cos_26', 'is_holiday']
for var in shared_vars:
    if var in coefs.index:
        sig = '***' if pvalues[var] < 0.01 else ('**' if pvalues[var] < 0.05 else ('*' if pvalues[var] < 0.10 else ''))
        print(f"  {var:15}: {coefs[var]:8.4f} {sig}")



PART 4: PPG-SPECIFIC ELASTICITIES

PPG-Specific Elasticities:
------------------------------------------------------------------------------------------
ppg_id  own_price_elasticity own_price_sig  cross_price_elasticity cross_price_sig  tdp_elasticity tdp_sig
PPG_01             -2.568681           ***                0.650316             ***        0.686885     ***
PPG_02             -2.448001           ***                0.545908             ***        0.718819     ***
PPG_03             -2.541433           ***                1.148859             ***        0.608244     ***
PPG_04             -2.451740           ***                0.958374             ***        0.640859     ***
PPG_05             -2.265507           ***                1.157095             ***        0.697315     ***
PPG_06             -2.535964           ***                0.617748             ***        0.772099     ***
PPG_07             -2.701672           ***                0.674816             ***        0.77317

In [13]:

print("\n" + "="*70)
print("PART 6: DEMAND SIMULATION")
print("="*70)

def simulate_demand_for_ppg(ppg_id, price_scenarios, model, df_model, 
                             comp_price=None, tdp=None, base_unit_pct=None):
    """
    Simulate demand for a single PPG across multiple price points.
    
    Parameters:
    -----------
    ppg_id : str
        The PPG to simulate (e.g., 'PPG_01')
    price_scenarios : list or array
        List of prices to test (e.g., [8.50, 9.00, 9.50, 10.00])
    model : statsmodels result
        Fitted OLS model
    df_model : DataFrame
        Original model data (to get baseline values)
    comp_price : float, optional
        Competitor price assumption. If None, uses current average.
    tdp : float, optional
        TDP assumption. If None, uses current average.
    base_unit_pct : float, optional
        Base unit % assumption. If None, uses current average.
    
    Returns:
    --------
    DataFrame with predictions for each price scenario
    """
    
    # Get baseline values for this PPG (most recent or average)
    ppg_data = df_model[df_model['ppg_id'] == ppg_id].copy()
    
    # Use provided values or defaults
    if comp_price is None:
        comp_price = ppg_data['comp_price'].iloc[-1]  # Most recent
    if tdp is None:
        tdp = ppg_data['tdp'].mean()
    if base_unit_pct is None:
        base_unit_pct = ppg_data['base_unit_pct'].mean()
    
    # Get most recent time features (for "next period" prediction)
    latest = ppg_data.iloc[-1]
    
    results = []
    for price in price_scenarios:
        # Create prediction row
        pred_row = pd.DataFrame({
            'ppg_id': [ppg_id],
            'ln_own_price': [np.log(price)],
            'ln_comp_price': [np.log(comp_price)],
            'ln_tdp': [np.log(tdp)],
            'base_unit_pct': [base_unit_pct],
            'trend': [latest['trend'] + 1],  # Next period
            'sin_52': [latest['sin_52']],
            'cos_52': [latest['cos_52']],
            'sin_26': [latest['sin_26']],
            'cos_26': [latest['cos_26']],
            'is_holiday': [0],  # Assume non-holiday
        })
        pred_row['ppg_id'] = pd.Categorical(pred_row['ppg_id'], categories=ppg_ids)
        
        # Predict
        ln_units_pred = model.predict(pred_row)[0]
        units_pred = np.exp(ln_units_pred)
        
        results.append({
            'ppg_id': ppg_id,
            'own_price': price,
            'comp_price': comp_price,
            'price_gap': price - comp_price,
            'tdp': tdp,
            'predicted_units': units_pred,
            'predicted_revenue': price * units_pred,
        })
    
    return pd.DataFrame(results)


# Example: Simulate for PPG_01
print("\nSimulation Example: PPG_01")
print("-" * 50)

# Define price range to test (±20% from current)
current_price = 9.49
price_range = np.linspace(current_price * 0.8, current_price * 1.2, 9)

sim_results = simulate_demand_for_ppg(
    ppg_id='PPG_01',
    price_scenarios=price_range,
    model=model,
    df_model=df_model
)

print(sim_results.to_string(index=False))


PART 6: DEMAND SIMULATION

Simulation Example: PPG_01
--------------------------------------------------
ppg_id  own_price  comp_price  price_gap       tdp  predicted_units  predicted_revenue
PPG_01     7.5920     9.30339   -1.71139 78.392954     80856.903594      613865.612082
PPG_01     8.0665     9.30339   -1.23689 78.392954     69196.871884      558176.567054
PPG_01     8.5410     9.30339   -0.76239 78.392954     59747.893063      510306.754654
PPG_01     9.0155     9.30339   -0.28789 78.392954     52000.455867      468810.109870
PPG_01     9.4900     9.30339    0.18661 78.392954     45581.246729      432566.031456
PPG_01     9.9645     9.30339    0.66111 78.392954     40212.183012      400694.297621
PPG_01    10.4390     9.30339    1.13561 78.392954     35683.019479      372495.040337
PPG_01    10.9135     9.30339    1.61011 78.392954     31832.651321      347405.640196
PPG_01    11.3880     9.30339    2.08461 78.392954     28536.114555      324969.272548


In [14]:
def simulate_all_ppgs(df_model, model, price_change_range=(-0.15, 0.15), n_points=7):
    """
    Run price simulations for all PPGs.
    
    Parameters:
    -----------
    price_change_range : tuple
        Min and max % price change to test (e.g., (-0.15, 0.15) for ±15%)
    n_points : int
        Number of price points to simulate
    
    Returns:
    --------
    DataFrame with all simulations
    """
    
    all_results = []
    
    for ppg_id in df_model['ppg_id'].unique():
        # Get current price for this PPG
        ppg_data = df_model[df_model['ppg_id'] == ppg_id]
        current_price = ppg_data['own_base_price'].iloc[-1]
        
        # Create price range
        price_range = np.linspace(
            current_price * (1 + price_change_range[0]),
            current_price * (1 + price_change_range[1]),
            n_points
        )
        
        # Simulate
        sim = simulate_demand_for_ppg(
            ppg_id=ppg_id,
            price_scenarios=price_range,
            model=model,
            df_model=df_model
        )
        sim['current_price'] = current_price
        sim['price_change_pct'] = (sim['own_price'] - current_price) / current_price * 100
        
        all_results.append(sim)
    
    return pd.concat(all_results, ignore_index=True)

In [15]:
full_simulation = simulate_all_ppgs(df_model, model)

print("\nSimulation Summary (showing subset):")
print(full_simulation[full_simulation['ppg_id'].isin(['PPG_01', 'PPG_02', 'PPG_03'])][
    ['ppg_id', 'own_price', 'price_change_pct', 'predicted_units', 'predicted_revenue']
].head(21).to_string(index=False))


Simulation Summary (showing subset):
ppg_id  own_price  price_change_pct  predicted_units  predicted_revenue
PPG_01   8.241657     -1.500000e+01     65482.040184      539680.521145
PPG_01   8.726460     -1.000000e+01     56540.329468      493396.949193
PPG_01   9.211264     -5.000000e+00     49208.813172      453275.360067
PPG_01   9.696067     -1.832039e-14     43134.219057      418232.285382
PPG_01  10.180871      5.000000e+00     38053.393342      387416.670860
PPG_01  10.665674      1.000000e+01     33767.377798      360151.839684
PPG_01  11.150477      1.500000e+01     30123.716524      335893.815713
PPG_02  11.623391     -1.500000e+01     40681.564348      472857.727142
PPG_02  12.307120     -1.000000e+01     35369.547075      435297.254411
PPG_02  12.990849     -5.000000e+00     30984.733370      402517.983720
PPG_02  13.674578      0.000000e+00     27328.458610      373705.127852
PPG_02  14.358306      5.000000e+00     24251.785686      348214.571473
PPG_02  15.042035      1.0

In [17]:
print("\n" + "="*70)
print("PART 8: COMPETITIVE SCENARIOS")
print("="*70)

def simulate_competitive_scenarios(ppg_id, own_price_scenarios, comp_scenarios, 
                                    model, df_model):
    """
    Create 2D simulation: own price × competitor price scenarios.
    """
    results = []
    
    ppg_data = df_model[df_model['ppg_id'] == ppg_id]
    current_comp_price = ppg_data['comp_price'].iloc[-1]
    
    for own_price in own_price_scenarios:
        for comp_scenario_name, comp_price in comp_scenarios.items():
            sim = simulate_demand_for_ppg(
                ppg_id=ppg_id,
                price_scenarios=[own_price],
                model=model,
                df_model=df_model,
                comp_price=comp_price
            )
            sim['comp_scenario'] = comp_scenario_name
            results.append(sim)
    
    return pd.concat(results, ignore_index=True)


# Example: PPG_01 with competitive scenarios
print("\nCompetitive Scenario Analysis: PPG_01")
print("-" * 60)

# Current competitor price
current_comp = df_model[df_model['ppg_id'] == 'PPG_01']['comp_price'].iloc[-1]

# Define scenarios
comp_scenarios = {
    'comp_drops_10%': current_comp * 0.90,
    'comp_drops_5%': current_comp * 0.95,
    'comp_stays': current_comp,
    'comp_raises_5%': current_comp * 1.05,
    'comp_matches_us': None,  # Will be set dynamically
}

# Own price scenarios
own_prices = [9.00, 9.49, 9.99, 10.49]

# Run simulation (simplified - without dynamic matching)
del comp_scenarios['comp_matches_us']  # Remove for simple example
comp_sim = simulate_competitive_scenarios(
    ppg_id='PPG_01',
    own_price_scenarios=own_prices,
    comp_scenarios=comp_scenarios,
    model=model,
    df_model=df_model
)

# Pivot for easier reading
comp_pivot = comp_sim.pivot_table(
    index='own_price', 
    columns='comp_scenario', 
    values='predicted_units',
    aggfunc='mean'
)
print("\nPredicted Units by Own Price × Competitor Scenario:")
print(comp_pivot.round(0).to_string())


PART 8: COMPETITIVE SCENARIOS

Competitive Scenario Analysis: PPG_01
------------------------------------------------------------

Predicted Units by Own Price × Competitor Scenario:
comp_scenario  comp_drops_10%  comp_drops_5%  comp_raises_5%  comp_stays
own_price                                                               
9.00                  48772.0        50517.0         53915.0     52231.0
9.49                  42563.0        44086.0         47051.0     45581.0
9.99                  37304.0        38638.0         41237.0     39949.0
10.49                 32905.0        34083.0         36375.0     35239.0


In [18]:
print("\n" + "="*70)
print("PART 9: EXPORT FOR MARGIN ANALYSIS")
print("="*70)

# Save elasticity summary
elasticity_df.to_csv('ppg_elasticities.csv', index=False)
print("Saved: ppg_elasticities.csv")

# Save full simulation
full_simulation.to_csv('price_simulation_results.csv', index=False)
print("Saved: price_simulation_results.csv")

# Save model coefficients
coef_df = pd.DataFrame({
    'coefficient': model.params,
    'std_error': model.bse,
    'pvalue': model.pvalues
})
coef_df.to_csv('model_coefficients.csv')
print("Saved: model_coefficients.csv")

print("\n" + "="*70)
print("NEXT STEPS")
print("="*70)
next_steps = """
1. ADD COGS DATA:
   - Join COGS to simulation results by PPG
   - Calculate: margin = (price - COGS) × predicted_units

2. FIND OPTIMAL PRICE:
   - For each PPG, find price that maximizes margin
   - Apply constraints (min margin %, price gaps, ladder logic)

3. SENSITIVITY ANALYSIS:
   - Run with different competitor assumptions
   - Calculate confidence intervals on predictions

4. VALIDATE:
   - Holdout test on last 13 weeks
   - Check predictions vs actuals before deploying

Example margin calculation:

    full_simulation['cogs'] = full_simulation['ppg_id'].map(cogs_dict)
    full_simulation['margin'] = (full_simulation['own_price'] - full_simulation['cogs']) * full_simulation['predicted_units']
    
    # Find optimal price per PPG
    optimal = full_simulation.loc[full_simulation.groupby('ppg_id')['margin'].idxmax()]
"""
print(next_steps)

print("\n" + "="*70)
print("DONE!")
print("="*70)



PART 9: EXPORT FOR MARGIN ANALYSIS
Saved: ppg_elasticities.csv
Saved: price_simulation_results.csv
Saved: model_coefficients.csv

NEXT STEPS

1. ADD COGS DATA:
   - Join COGS to simulation results by PPG
   - Calculate: margin = (price - COGS) × predicted_units

2. FIND OPTIMAL PRICE:
   - For each PPG, find price that maximizes margin
   - Apply constraints (min margin %, price gaps, ladder logic)

3. SENSITIVITY ANALYSIS:
   - Run with different competitor assumptions
   - Calculate confidence intervals on predictions

4. VALIDATE:
   - Holdout test on last 13 weeks
   - Check predictions vs actuals before deploying

Example margin calculation:

    full_simulation['cogs'] = full_simulation['ppg_id'].map(cogs_dict)
    full_simulation['margin'] = (full_simulation['own_price'] - full_simulation['cogs']) * full_simulation['predicted_units']

    # Find optimal price per PPG
    optimal = full_simulation.loc[full_simulation.groupby('ppg_id')['margin'].idxmax()]


DONE!
